In [24]:
import re

import bibtexparser # type: ignore
import bibtexparser.middlewares as m # type: ignore

# %pip install "bibtexparser==2.0.0b8"


STOP_WORDS = [
    'Retrieved', 'Available from', 'Available at', 'Accessed at',
    'https://', 'http://', 'doi', 'DOI', 'arXiv', 'arxiv',
]


def normalize_whitespace(text: str) -> str:
    """Collapse repeated whitespace into single spaces and trim ends."""
    return re.sub(r'\s+', ' ', text).strip()


def extract_url(citation: str) -> str:
    """Grab the last HTTP/HTTPS URL in the citation."""
    urls = re.findall(r'https?://\S+', citation)
    return urls[-1].rstrip('.,);]') if urls else ''


def clean_authors(raw: str) -> str:
    """Tidy the authors segment but preserve initials and punctuation."""
    cleaned = raw.rstrip(' (')
    return normalize_whitespace(cleaned)


def extract_title(post_year: str) -> str:
    """
    Derive the title from the portion of text that follows the year.
    Strategy:
      1. Skip the first sentence (often date/location).
      2. Stop at the first stop word (Retrieved, doi, http, etc.).
      3. Take the first sentence from the remaining text.
    """
    if '.' in post_year:
        post_year = post_year[post_year.find('.') + 1:]
    title_candidate = post_year.strip()

    stop_idx = len(title_candidate)
    lowered = title_candidate.lower()
    for stop_word in STOP_WORDS:
        idx = lowered.find(stop_word.lower())
        if idx != -1 and idx < stop_idx:
            stop_idx = idx
    title_candidate = title_candidate[:stop_idx].strip()

    sentence_end = re.search(r'[.!?]', title_candidate)
    if sentence_end:
        title_candidate = title_candidate[:sentence_end.end()]

    title_candidate = title_candidate.strip(' "\'“”')
    title = re.sub(r'[.!?]+$', '', title_candidate).strip()
    return title


def parse_citation(citation: str) -> dict[str, str]:
    citation = normalize_whitespace(citation)
    year_match = re.search(r'\b(1[89]\d{2}|20\d{2}|21\d{2})\b', citation)
    if not year_match:
        raise ValueError(f"Could not locate a year in citation: {citation}")

    authors_raw = citation[:year_match.start()].strip()
    authors = clean_authors(authors_raw)
    year = year_match.group(0)

    post_year = citation[year_match.end():].strip()
    title = extract_title(post_year)
    url = extract_url(citation)

    return {
        'authors': authors,
        'title': title,
        'year': year,
        'url': url,
    }
    
def parse_with_bibtexparser(text: str) -> dict[str, str]:
    lib = bibtexparser.parse_string(text)  # , append_middleware=[m.SeparateCoAuthors()]
    assert len(lib.blocks) == 1
    block = lib.blocks[0]
    parsed = { # type: ignore
        'authors': block['author'] if 'author' in block else '', # type: ignore
        'title': block['title'].removeprefix('{').removesuffix('}') if 'title' in block else '', # type: ignore
        'year': block['year'] if 'year' in block else '', # type: ignore
        'url': block['url'] if 'url' in block else '', # type: ignore
    }

    return parsed # type: ignore

def parse_header(text: str) -> dict[str, str]:
    if text.lstrip().startswith('@'):
        return parse_with_bibtexparser(text)
    else:
        return parse_citation(text)

In [25]:
import json
from pathlib import Path

from tqdm.auto import tqdm


data = json.loads(Path('papers.json').read_text())

parsed: list[dict[str, str]] = []

for row in tqdm(data):
    row['header_parsed'] = parse_header(row['header'])

  0%|          | 0/810 [00:00<?, ?it/s]

In [26]:
Path('papers_parsed.json').write_text(json.dumps(data, ensure_ascii=False, indent=4))

1659770

In [27]:
import pandas as pd
# %pip install openpyxl

pd.DataFrame([row['header_parsed'] for row in data]).to_excel('papers.xlsx') # type: ignore

In [ ]:
# lib = bibtexparser.parse_string('@article{Bajaj2016Nov,\n\tauthor = {Bajaj, Payal and Campos, Daniel and Craswell, Nick and Deng, Li and Gao, Jianfeng and Liu, Xiaodong and Majumder, Rangan and McNamara, Andrew and Mitra, Bhaskar and Nguyen, Tri and Rosenberg, Mir and Song, Xia and Stoica, Alina and Tiwary, Saurabh and Wang, Tong},\n\ttitle = {{MS MARCO: A Human Generated MAchine Reading COmprehension Dataset}},\n\tjournal = {arXiv},\n\tyear = {2016},\n\tmonth = nov,\n\teprint = {1611.09268},\n\tdoi = {10.48550/arXiv.1611.09268}\n}')  # , append_middleware=[m.SeparateCoAuthors()]
# assert len(lib.blocks) == 1
# block = lib.blocks[0]
# parsed = { # type: ignore
#     'authors': block['author'] if 'author' in block else '', # type: ignore
#     'title': block['title'].removeprefix('{').removesuffix('}') if 'title' in block else '', # type: ignore
#     'year': block['year'] if 'year' in block else '', # type: ignore
#     'url': block['url'] if 'url' in block else '', # type: ignore
# }